In [1]:
import librosa
import numpy as np

y, sr = librosa.load("./input/ILFTD_Mono.wav")

# STFT
D = np.abs(librosa.stft(y))

In [2]:
# pitches, magnitudes = librosa.piptrack(y=y, sr=sr)

In [3]:
# pitch_track = []

# for i in range(pitches.shape[1]):
#     index = magnitudes[:, i].argmax()
#     pitch = pitches[index, i]
#     pitch_track.append(pitch)

In [4]:
onset_frames = librosa.onset.onset_detect(
    y=y,
    sr=sr,
    hop_length=512
)

onset_times = librosa.frames_to_time(onset_frames, sr=sr)

segments = []

for i in range(len(onset_times) - 1):
    start = onset_times[i]
    end = onset_times[i + 1]
    segments.append((start, end))


notes = []

for start, end in segments:

    start_sample = int(start * sr)
    end_sample = int(end * sr)

    segment = y[start_sample:end_sample]

    pitch = librosa.yin(
        segment,
        fmin=librosa.note_to_hz("C2"),
        fmax=librosa.note_to_hz("C7")
    )

    freq = np.median(pitch)

    notes.append((start, end, freq))

In [5]:
import math

def freq_to_note(freq):
    if freq <= 0:
        return None

    note_num = 69 + 12 * math.log2(freq / 440.0)
    note_num = round(note_num)

    names = ['C','C#','D','D#','E','F',
             'F#','G','G#','A','A#','B']

    note = names[note_num % 12]
    octave = note_num // 12 - 1

    return f"{note}{octave}"

def freq_to_midi(freq):
    import math
    if freq <= 0:
        return None
    return int(round(69 + 12 * math.log2(freq / 440.0)))

In [6]:
note_results = []

for start, end, freq in notes:
    note = freq_to_midi(freq)
    duration = end - start
    if duration < 0.05:
        continue

    note_results.append((note, start, duration))

In [7]:
import pretty_midi

def notes_to_midi(note_data, output_file="output.mid"):
    """
    note_data: list of (start, end, freq)
    """

    midi = pretty_midi.PrettyMIDI()
    
    # Create an instrument (Acoustic Grand Piano)
    instrument = pretty_midi.Instrument(
        program=pretty_midi.instrument_name_to_program("Acoustic Grand Piano")
    )

    for start, end, freq in note_data:
        midi_note = freq_to_midi(freq)
        
        if midi_note is None:
            continue
        
        note = pretty_midi.Note(
            velocity=100,           # volume (0–127)
            pitch=midi_note,
            start=start,
            end=end
        )
        
        instrument.notes.append(note)

    midi.instruments.append(instrument)
    midi.write(output_file)

In [8]:
notes_to_midi(notes, "./output/transcription.mid")